<a href="https://colab.research.google.com/github/Text-Machine/mask-predict/blob/main/1-compute-gradients.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> </a>

# Compute Gradients

This notebooks is prepared for computing gradients given a preformatted csv file.

Gradients compute the relation between context words and a given target word, which in our case is the token we masked, and which we want to investigate. This masked token is also referred to as the target token.

This notebooks is written for Colab but should also work on other platforms with some minor adjustments.

To measure the influence between context and target words, we need a sentences with a [MASKED] token, and a set of target tokens, for which we want to compute the gradients in the masked token position.

There are two main options strategies, and these options relate to how we model the set of target tokens

## Predictions

Here we model the relationship between contexts words and tokens predicted by a BERT model using in the fill-mask pipeline. 

### Predictions-filtered

This is a subtask, where we narrow down the set of target tokens by thematically filtering the predicted words, for example, only keep examples where BERT predicted human-like words. This filtering can happen iteratively.

## Contrastive

We provide a contrastive pair of keywords, e.g. 'machine' and 'slave' and for each senteces we compute gradients for the same pair.



In [ ]:
#!git clone https://github.com/Text-Machine/mask-predict.git

In [ ]:
#%cd mask-predict

In [ ]:
#!pip install  -q -e .

# Prepare variables and load data

In [ ]:
from explain import *
from collections import Counter
import pandas as pd
from pathlib import Path
import json

In [2]:
collection,genre_suffix = 'blb',''
if collection == 'blb':
  genre_suffix = '_with_genre'

maskedToken = 'machine'

modelName = "Livingwithmachines/bert_1760_1900"
dataPath = 'input_data'
processedFolder = Path('gradient_data')
processedFolder.mkdir(exist_ok=True, parents=True)
predCol = "pred_bert_1760_1900"
iloc_range = (0, -1)

In [29]:
#%cd /content

In this notebook we analyse the newspaper sentences where machine in mentioned by BLERT predicts words related to slavery.

In [ ]:
#!unzip -o "{collection}_{maskedToken}_clusters{genre_suffix}_deduplicated.tsv.zip"

We rank sentences by their similarity to the 'slave' cluster based on the BLERT predictions. 

In [4]:

data_df = pd.read_csv(f"./{dataPath}/{collection}_{maskedToken}_clusters{genre_suffix}_deduplicated.tsv", sep="\t", index_col=0)
data_df.head()

,article_path,identifier,date,shelfmarks,publisher,title,edition,contributors,creators,prevSentence,...,slave_1760_1900,artisan_1760_1900,woman_1760_1900,machine_contemporary,boy_contemporary,girl_contemporary,slave_contemporary,artisan_contemporary,woman_contemporary,genre
0,005009643_01_text.json,5009643,1835,British Library MAPS 7107.d.19./British Librar...,J.S. and C. Adams,"Report on the geology, mineralogy, botany, and...","2nd ed, corrected and enlarged.",NaN,"Hitchcock, Edward.",I deduce another moral consideration of no lit...,...,0.263,0.369,0.295,0.124,0.111,0.128,0.105,0.085,0.144,Non-fiction
1,005009643_01_text.json,5009643,1835,British Library MAPS 7107.d.19./British Librar...,J.S. and C. Adams,"Report on the geology, mineralogy, botany, and...","2nd ed, corrected and enlarged.",NaN,"Hitchcock, Edward.",So constant and uniform are the operations of ...,...,0.224,0.219,0.209,0.110,0.143,0.152,0.109,0.098,0.151,Non-fiction
2,000382552_01_text.json,382552,1859,British Library HMNTS 9525.b.17.,J. Blacket,The Two Battles of Newbury. Being the substanc...,NaN,NaN,"BLUNDELL, Bezer.","6 This was no vaunt , — for Cromwell never boa...",...,0.158,0.160,0.165,0.097,0.087,0.076,0.078,0.089,0.090,Non-fiction
3,000302544_01_text.json,302544,1889,British Library HMNTS 012633.m.31.,NaN,"To Call Her Mine, etc. (Katharine Regina.-'Sel...",NaN,NaN,"Besant, Walter - Sir","They have , therefore , ascertained the very l...",...,0.144,0.153,0.173,0.111,0.129,0.116,0.114,0.119,0.142,Fiction
4,000302544_01_text.json,302544,1889,British Library HMNTS 012633.m.31.,NaN,"To Call Her Mine, etc. (Katharine Regina.-'Sel...",NaN,NaN,"Besant, Walter - Sir",Ho ' The English clerks are sent away because ...,...,0.129,0.112,0.178,0.057,0.079,0.073,0.075,0.071,0.084,Fiction


In [5]:
data_df.tail(3)

,article_path,identifier,date,shelfmarks,publisher,title,edition,contributors,creators,prevSentence,...,slave_1760_1900,artisan_1760_1900,woman_1760_1900,machine_contemporary,boy_contemporary,girl_contemporary,slave_contemporary,artisan_contemporary,woman_contemporary,genre
77904,002591238_01_text.json,2591238,1877,British Library HMNTS 12270.i.1.,NaN,The Prose and Poetry of Ireland. A choice coll...,NaN,NaN,"MURRAY, John O'Kane.",TO SIE JOSHUA EEYNOLDS .,...,0.188,0.162,0.190,0.227,0.190,0.191,0.185,0.191,0.201,Fiction
77905,002591238_01_text.json,2591238,1877,British Library HMNTS 12270.i.1.,NaN,The Prose and Poetry of Ireland. A choice coll...,NaN,NaN,"MURRAY, John O'Kane.",Yet you continue to support an administration ...,...,0.255,0.219,0.198,0.112,0.112,0.125,0.167,0.139,0.147,Fiction
77906,010219361_02_text.json,10219361,1851,British Library HMNTS 9055.c.1./British Librar...,R. Bentley,"A year on the Punjab frontier, in 1848-49",NaN,NaN,"Edwardes, Herbert B. (Herbert Benjamin) - Sir","If they do not , the mildest youth who fills t...",...,0.135,0.190,0.165,0.385,0.136,0.132,0.142,0.176,0.169,Non-fiction


We get the first 1000 sentences, and the top 5 predictions. This means we only look at context words where our target words of interest (like 'slave' appear in the top five words).

# Thematic filtering of the predicted target

To minimise the computation we focus on a specific sample of predictions, i.e. occurances where the MLM predicted human type words instead of machine words. 

We start with a seed list of predictions, look which other words are predicted together with these seedwords and use this expanded list as our main target of investigation.

In [23]:
texts, predicted_targets = build_texts_targets(
    data_df, start=iloc_range[0], end=iloc_range[1],
    pred_col=predCol, top_n=5
)

In [24]:
def filter_texts_targets(texts, predicted_targets, kw_filters, data_df, only_keywords=False):
    filtered_texts = []
    filtered_predicted_targets = []
    filter_identifiers = []
    for i,(text, preds) in enumerate(zip(texts, predicted_targets)):
        if set(preds).intersection(set(kw_filters)):
            filtered_texts.append(text)
            filter_identifiers.append(i)
            if only_keywords:
                preds = [p for p in preds if p in kw_filters]
                if not preds:
                    continue
            filtered_predicted_targets.append([data_df.targetExpression.iloc[i]]+preds)
    return filter_identifiers, filtered_texts, filtered_predicted_targets

In the example we first fetch human words by looking which other prediction co-occur with a set of the seed words such 'child', 'man' etc. The idea is the obtain an expanded list of words to investigate, and ensure our sample isn't completely biased by the 
We use `filter_texts_targets`, in the first round we set the  `only_keywords` argument to `False`, in the second round to `True` because we want to work only with the expanded wordlist.

In [25]:
#kw_filters = ['child', 'children', 'woman', 'man', 'women', 'men', 'slave','slaves']
kw_filters = open(f'{dataPath}/250_freq_pred_KB_edit.txt').read().splitlines()
kw_filters[:4]

['man', 'men', 'woman', 'child']

In [27]:

print('Before filtering:', len(texts), len(predicted_targets))
filter_identifiers, texts, predicted_targets = filter_texts_targets(texts, 
                                                                    predicted_targets, 
                                                                    kw_filters, data_df,
                                                                    only_keywords=True)
print('After filtering:', len(texts), len(predicted_targets))

Before filtering: 77906 77906
After filtering: 19003 19003


In [ ]:
data_df.iloc[filter_identifiers].to_csv(f"{processedFolder}/{collection}_{maskedToken}_filtered{genre_suffix}_kw_filtered.tsv", sep="\t")


In [ ]:
# from google.colab import files
# files.download(f"{processedFolder}/{collection}_{maskedToken}_filtered{genre_suffix}_kw_filtered.tsv") 

In [29]:
Counter([t for l in predicted_targets for t in l]).most_common(10)

[('machine', 13625),
 ('machines', 5066),
 ('man', 2819),
 ('blacksmith', 2404),
 ('men', 2269),
 ('body', 1476),
 ('hand', 1305),
 ('press', 909),
 ('people', 874),
 ('hands', 820)]

In [22]:
open(f'{dataPath}/250_freq_pred.txt','w').write('\n'.join([s for s,i in Counter([t for l in predicted_targets for t in l]).most_common(250)]))

1708

In [32]:
idx = 100
texts[idx],predicted_targets[idx], filter_identifiers[idx]

("'; The [MASK] Martha Gunn superintended were always the most in request , for she was immensely popular with bathers .",
 ['machines', 'ladies', 'women', 'girls', 'people'],
 377)

We run the explainer using BLERT as the model we inspect, our targets are the top 5 predictions, the texts are the 1000 sentences selected in the previous cell.

# Compute Integrated Gradients.

In [19]:
explainer = MaskedLMExplainer(model_name=modelName, device=pick_device())

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: Livingwithmachines/bert_1760_1900
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
results = explainer.explain(texts, predicted_targets, word_agg="max")

In [ ]:
with open(f'results_{collection}_{maskedToken}_pred.json', 'w') as f:
    json.dump(results, f)

In [ ]:
# from google.colab import files
# files.download(f'results_{collection}_{maskedToken}_pred_kw_filtered.json') 

Only retain results for context word contribution when slave or machine are predicted by BLERT.

In [ ]:
results_df = result_as_dataframe(results, kw_filters)
results_df['identifier'] = results_df.id.apply(lambda x: filter_identifiers[int(x)])
results_df.to_csv(f'results_{collection}_{maskedToken}_pred_kw_filtered.csv')

In [ ]:
#files.download(f'results_{collection}_{maskedToken}_pred_kw_filtered.csv') 

# Contrastive Analysis

Contrastive analysis, we use slave and machine as target words for all sentences, ignoring the actual predictions.

In [10]:
if maskedToken == 'slave':
      contrastive_dict = {'slave':'machine', 'slaves':'machines'}
elif maskedToken == 'machine':
    contrastive_dict = {'machine':'slave', 'machines':'slaves'}

data_df['targetExpressionProccessed'] = data_df['targetExpression'].apply(lambda x: x.lower().strip())
data_df['targetContrExpressionProccessed'] = data_df['targetExpressionProccessed'].apply(lambda x: contrastive_dict.get(x, x))
contrastive_targets = data_df.iloc[iloc_range[0]:iloc_range[1]][['targetContrExpressionProccessed', 'targetExpressionProccessed']].values.tolist()

len(texts) == len(contrastive_targets)

True

In [ ]:
contrastive_targets[:3]

[['machine', 'slave'],
 ['machines', 'slaves'],
 ['machines', 'slaves'],
 ['machine', 'slave'],
 ['machine', 'slave'],
 ['machine', 'slave'],
 ['machine', 'slave'],
 ['machine', 'slave'],
 ['machine', 'slave'],
 ['machines', 'slaves']]

In [ ]:
results2 = explainer.explain(texts, contrastive_targets, word_agg="max")

In [ ]:
with open(f'results_{collection}_{maskedToken}_constrastive.json', 'w') as f:
    json.dump(results2, f)

In [12]:
with open(f'{processedFolder}/results_{collection}_{maskedToken}_constrastive.json', 'r') as f:
    results2 = json.load(f)

In [13]:
unique = list(set(x for sublist in contrastive_targets for x in sublist))
unique

['machine', 'slaves', 'slave', 'machines']

In [14]:
results_df = result_as_dataframe(results2,unique)
results_df.shape

100%|██████████| 53726/53726 [10:55<00:00, 81.93it/s]  


(2312741, 4)

In [15]:
results_df.to_csv(f'{processedFolder}/results_{collection}_{maskedToken}_constrastive.csv')

# Fin.